<a href="https://colab.research.google.com/github/Ecc535/Mental_Health_LLM_Benchmarking/blob/main/Flan_T5_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers accelerate torch

In [2]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

In [3]:
model_id = "google/flan-t5-xl"
print(f"Loading {model_id}...")

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl", device_map="auto")

Loading google/flan-t5-xl...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
FEATURE_GROUPS = {
    "Rhythm": [
        "steps_48phi", "steps_8phi", "steps_psd_auc_64h",
        "steps_12p_reject", "steps_128p_reject", "steps_IV"
    ],
    "Heart Rate Variability": [
        "hrstd_max", "hrstd_std", "hrmax_std", "hrmean_max",
        "hrmax_mean", "hrentropy_std", "RestingHeartRate_std", "hrmax_max"
    ],
    "Shift Pattern": [
        "shift_hours_median", "shift_type_shannon_entropy",
        "shift_hours_shannon_entropy"
    ],
    "Inactivity at Daytime": [
        "duration_entropy_non_step_std"
    ],
    "Morning Energy": [
        "happiness_morning_std", "happiness_morning_shannon_entropy",
        "sleepiness_daytime_shannon_entropy", "energy_morning_std",
        "alterness_morning_std", "happiness_morning_median"
    ],
    "Evening Energy": [
        "energy_evening_median", "relax_evening_median",
        "health_evening_std", "energy_evening_std",
        "health_evening_median", "happiness_evening_median",
        "happiness_evening_std"
    ],
    "Caffeine Intake": [
        "caffeine_cups_shannon_entropy",
        "awake_type_shannon_entropy",
        "awake_type_std",
        "awake_type_median"
    ],
    "Sleep": [
        "sleep_efficiency_median",
        "sleep_duration_max",
        "nap_duration_fitbit_max"
    ]
}

In [5]:
def build_prompt(grouped_dict):
    formatted_sections = []

    for group, features in grouped_dict.items():
        section = f"## {group}\n"
        for k, v in features.items():
            section += f"{k}: {v}\n"
        formatted_sections.append(section)

    data_block = "\n".join(formatted_sections)

    prompt = f"""
Data:
{data_block}

Write a concise professional summary describing:
- Circadian rhythm stability
- Cardiovascular variability
- Work/shift regularity
- Daytime inactivity
- Morning and evening energy patterns
- Sleep quality

Interpret patterns at a high level.
Do NOT list features individually.
Do NOT invent values.

Summary:
"""
    return prompt.strip()

In [13]:
def build_classification_prompt(summary):
    return f"""Read the following behavioral health summary and classify the burnout risk.

Summary:
{summary}

Question: Based on the summary, is the burnout risk level High or Low?
Answer:"""

In [7]:
ground_truth = pd.read_csv("/content/jbs_scores_post.csv")
ground_truth.rename(columns={"u_id": "ID"}, inplace=True)
ground_truth["ID"] = ground_truth["ID"].astype(str)

In [8]:
data = pd.read_json("/content/all_participants_features.json")
data.head()

,1002,1102,1103,1105,1108,1115,1150,1151,1153,1154,...,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316
Caffeine Intake,"{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.5, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...",...,"{'awake_type_median': 1.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon...","{'awake_type_median': 0.0, 'awake_type_shannon..."
Evening Energy,"{'energy_evening_median': 24.0, 'energy_evenin...","{'energy_evening_median': 41.0, 'energy_evenin...","{'energy_evening_median': 68.0, 'energy_evenin...","{'energy_evening_median': 80.0, 'energy_evenin...","{'energy_evening_median': 28.0, 'energy_evenin...","{'energy_evening_median': 70.5, 'energy_evenin...","{'energy_evening_median': 30.0, 'energy_evenin...","{'energy_evening_median': 35.0, 'energy_evenin...","{'energy_evening_median': 43.0, 'energy_evenin...","{'energy_evening_median': 31.0, 'energy_evenin...",...,"{'energy_evening_median': 16.0, 'energy_evenin...","{'energy_evening_median': 36.0, 'energy_evenin...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 30.0, 'energy_evenin...","{'energy_evening_median': 10.0, 'energy_evenin...","{'energy_evening_median': 12.5, 'energy_evenin...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening...","{'energy_evening_median': 0.0, 'energy_evening..."
Heart Rate Variability,"{'RestingHeartRate_std': 3.201, 'hrentropy_std...","{'RestingHeartRate_std': 2.171, 'hrentropy_std...","{'RestingHeartRate_std': 1.956, 'hrentropy_std...","{'RestingHeartRate_std': 1.687, 'hrentropy_std...","{'RestingHeartRate_std': 2.103, 'hrentropy_std...","{'RestingHeartRate_std': 1.6520000000000001, '...","{'RestingHeartRate_std': 2.318, 'hrentropy_std...","{'RestingHeartRate_std': 1.324, 'hrentropy_std...","{'RestingHeartRate_std': 2.082, 'hrentropy_std...","{'RestingHeartRate_std': 1.974, 'hrentropy_std...",...,"{'RestingHeartRate_std': 2.659, 'hrentropy_std...","{'RestingHeartRate_std': 4.312, 'hrentropy_std...","{'RestingHeartRate_std': 3.65, 'hrentropy_std'...","{'RestingHeartRate_std': 3.229, 'hrentropy_std...","{'RestingHeartRate_std': 2.304, 'hrentropy_std...","{'RestingHeartRate_std': 1.867, 'hrentropy_std...","{'RestingHeartRate_std': 2.96, 'hrentropy_std'...","{'RestingHeartRate_std': 2.972, 'hrentropy_std...","{'RestingHeartRate_std': 3.002, 'hrentropy_std...","{'RestingHeartRate_std': 3.3, 'hrentropy_std':..."
Inactivity at Daytime,{'duration_entropy_non_step_std': 0.784},{'duration_entropy_non_step_std': 0.242},{'duration_entropy_non_step_std': 0.224},{'duration_entropy_non_step_std': 0.258},{'duration_entropy_non_step_std': 0.331},{'duration_entropy_non_step_std': 0.354},{'duration_entropy_non_step_std': 0.304},{'duration_entropy_non_step_std': 0.267},{'duration_entropy_non_step_std': 0.268},{'duration_entropy_non_step_std': 0.3460000000...,...,{'duration_entropy_non_step_std': 0.384},{'duration_entropy_non_step_std': 0.544},{'duration_entropy_non_step_std': 0.303},{'duration_entropy_non_step_std': 0.438},{'duration_entropy_non_step_std': 0.335},{'duration_entropy_non_step_std': 0.242},{'duration_entropy

In [14]:
participant_ids = list(data.columns)
batch_size = 4
results = []

print(f"\nProcessing {len(participant_ids)} participants in batches of {batch_size}...")

for i in tqdm(range(0, len(participant_ids), batch_size)):
    batch_ids = participant_ids[i:i + batch_size]
    batch_series = [data[pid] for pid in batch_ids]

    # --- STAGE 1: Summarization ---
    summary_prompts = [build_prompt(series) for series in batch_series]

    summary_inputs = tokenizer(
        summary_prompts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to("cuda")

    # torch.no_grad() prevents memory tracking, saving VRAM
    with torch.no_grad():
        summary_outputs = model.generate(
            **summary_inputs,
            max_new_tokens=250,
            temperature=0.3,
            do_sample=True
        )

    summaries = tokenizer.batch_decode(summary_outputs, skip_special_tokens=True)

    # --- STAGE 2: Classification ---
    class_prompts = [build_classification_prompt(summary) for summary in summaries]

    class_inputs = tokenizer(
        class_prompts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        class_outputs = model.generate(
            **class_inputs,
            max_new_tokens=3,
            temperature=0.0, # reasoning off
            do_sample=False
        )

    classifications = tokenizer.batch_decode(class_outputs, skip_special_tokens=True)

    # --- Save Batch Results ---
    for pid, summary, label in zip(batch_ids, summaries, classifications):
        results.append({
            "ID": pid,
            "llm_summary": summary,
            "llm_burnout_label": label.strip()
        })

df_results = pd.DataFrame(results)
df_results.to_csv("t5_llm_predictions.csv", index=False)
print("\nPredictions saved to t5_llm_predictions.csv!")


Processing 44 participants in batches of 4...


100%|██████████| 11/11 [01:42<00:00,  9.28s/it]


Predictions saved to t5_llm_predictions.csv!


In [15]:
print("\nRunning Evaluation...")
ground_truth = pd.read_csv("/content/jbs_scores_post.csv")

df_results["ID"] = df_results["ID"].astype(str)
ground_truth.rename(columns={"u_id": "ID"}, inplace=True)
ground_truth["ID"] = ground_truth["ID"].astype(str)

df_eval = df_results.merge(ground_truth, on="ID", how="inner")
print(f"Merged shape: {df_eval.shape}")


def clean_label(label):
    label = str(label).capitalize()
    if "High" in label: return "High"
    if "Low" in label: return "Low"
    return "Unknown"

df_eval["llm_burnout_label"] = df_eval["llm_burnout_label"].apply(clean_label)
df_eval["risk"] = df_eval["risk"].map({1: "High", 0: "Low"})

# Calculate Metrics
accuracy = accuracy_score(df_eval["risk"], df_eval["llm_burnout_label"])
print(f"\nAccuracy: {accuracy}")
print("\nDetailed Report:")
print(classification_report(df_eval["risk"], df_eval["llm_burnout_label"]))


Running Evaluation...
Merged shape: (44, 10)

Accuracy: 0.5909090909090909

Detailed Report:
              precision    recall  f1-score   support

        High       0.92      0.58      0.71        38
         Low       0.20      0.67      0.31         6

    accuracy                           0.59        44
   macro avg       0.56      0.62      0.51        44
weighted avg       0.82      0.59      0.65        44

